[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PTB-MR/mrseq/blob/main/examples/radial3d_flash.ipynb)


# 3D radial FLASH

Qualitative 3D imaging with a 3D radial (kooshball) FLASH (spoiled gradient echo) sequence using a golden means spoke distribution.

### Imports

In [ ]:
import importlib

if not importlib.util.find_spec('mrseq'):
    %pip install mrseq[examples]

In [ ]:
import tempfile
import time
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
from einops import rearrange
from mrpro.algorithms.reconstruction import DirectReconstruction
from mrpro.data import KData
from mrpro.data.traj_calculators import KTrajectoryIsmrmrd
from mrpro.operators.models import SpoiledGRE
from raw2ismrmrd.utils import combine_ismrmrd_files

from mrseq.sequences.radial3d_flash import main as create_seq
from mrseq.utils import sys_defaults

### Settings
We are going to use a small numerical phantom with an isotropic matrix size of 32 x 32 x 32 to reduce run times.
To ensure we really acquire all data in the steady state, we use a large number of dummy excitations before the actual image acquisition.

In [ ]:
image_matrix_size = [32, 32, 32]  # x,y,z
flip_angle_degree = 12
n_dummy_excitations = 200

tmp = tempfile.TemporaryDirectory()
fname_mrd = Path(tmp.name) / 'radial3d_flash.mrd'

### Create the digital phantom

We use the Brainweb phantom from [MRzero](https://github.com/MRsources/MRzero-Core), interpolated to our desired matrix size, with uniform B1 field.

In [ ]:
# Load 3D brainweb phantom
if not Path('subject05.npz').exists():
    urlretrieve(
        'https://github.com/MRsources/MRzero-Core/raw/main/documentation/playground_mr0/subject05.npz',
        'subject05.npz',
    )

obj_p = mr0.VoxelGridPhantom.brainweb('subject05.npz')
phantom = obj_p.interpolate(*image_matrix_size)
phantom.B1[:] = 1.0

### Create the 3D radial FLASH sequence

To create the 3D radial FLASH sequence, we use the previously imported [radial3d_flash script](../src/mrseq/sequences/radial3d_flash.py).

In [ ]:
sequence, fname_seq = create_seq(
    system=sys_defaults,
    test_report=False,
    timing_check=False,
    n_dummy_excitations=n_dummy_excitations,
    rf_flip_angle=flip_angle_degree,
    fov=float(phantom.size.numpy()[0]),
    n_readout=image_matrix_size[0],
    spoke_type='full',
    distribution='golden',
    excitation='block',
)

### Simulate the sequence

Now, we pass the sequence and the phantom to the MRzero simulation and save the simulated signal as an (ISMR)MRD file.

In [ ]:
mr0_sequence = mr0.Sequence.import_file(str(fname_seq.with_suffix('.seq')))
tstart = time.time()
signal, ktraj_adc = mr0.util.simulate(mr0_sequence, phantom, accuracy=1e-1)
print(f'Simulation time: {(time.time() - tstart) / 60:.2f} min')
mr0.sig_to_mrd(fname_mrd, signal, sequence)
combine_ismrmrd_files(fname_mrd, Path(f'{fname_seq}_header.h5'))

### Reconstruct the image
We use [MRpro](https://github.com/PTB-MR/MRpro) for the image reconstruction.

In [ ]:
kdata = KData.from_file(str(fname_mrd).replace('.mrd', '_with_traj.mrd'), trajectory=KTrajectoryIsmrmrd())
# check that the trajectory is 3D
assert (kdata.traj.kx != 0).any() and (kdata.traj.ky != 0).any() and (kdata.traj.kz != 0).any()
recon = DirectReconstruction(kdata, csm=None)
idata = recon(kdata)
assert idata.data.shape[-3:] == tuple(image_matrix_size)

### Compare to theoretical model

We can now compare the result to a simulation using the idealized signal model for spoiled gradient echo sequences. First, we need to calculate $T2^*$ using $1/T2^* = 1/T2 + 1/T2'$.

In [ ]:
t2star = 1 / (1 / phantom.T2 + 1 / phantom.T2dash)

In [ ]:
model = SpoiledGRE(flip_angle=np.deg2rad(flip_angle_degree), echo_time=kdata.header.te, repetition_time=kdata.header.tr)
idat_model = model(m0=phantom.PD, t1=phantom.T1, t2star=t2star)[0]
idat_model = idat_model.detach().abs().numpy().squeeze()
idat_model /= idat_model.max()
idat_model = rearrange(idat_model, 'x y z -> z y x')

# mask out background noise
obj_mask = np.zeros_like(idat_model)
obj_mask[idat_model > 0.01] = 1

idat = idata.data.abs().squeeze().numpy()
idat *= obj_mask
idat /= np.percentile(idat, 99.9)

# central axial slice
z_mid = idat.shape[0] // 2
fig, ax = plt.subplots(1, 3, figsize=(15, 3))
for cax in ax:
    cax.set_xticks([])
    cax.set_yticks([])

im = ax[0].imshow(idat_model[z_mid, ::-1, :], vmin=0, vmax=1, cmap='gray')
fig.colorbar(im, ax=ax[0], label='Model image')

im = ax[1].imshow(idat[z_mid, ::-1, :], vmin=0, vmax=1, cmap='gray')
fig.colorbar(im, ax=ax[1], label='MRseq image')

im = ax[2].imshow(idat[z_mid, ::-1, :] - idat_model[z_mid, ::-1, :], cmap='bwr', vmin=-0.1, vmax=0.1)
fig.colorbar(im, ax=ax[2], label='Difference')

relative_error = np.sum(np.abs(idat - idat_model)) / np.sum(np.abs(idat_model))
print(f'Relative error {relative_error}')
assert relative_error < 0.15